# Buổi 4 - Kết quả thực nghiệm DeepSolo + TrOCR

Notebook này dùng để xem kết quả A/B của Buổi 4 ngay trong `docs/`.

---

## Mục lục nhanh — **demo 1 ảnh cho ở đâu?**

| Bạn cần | Ở đâu trong repo / notebook |
|--------|------------------------------|
| **File notebook** | `docs/buoi-4-ket-qua-thuc-nghiem-deepsolo-trocr.ipynb` (đường đầy đủ: `D:\ComputerVisionNew\docs\buoi-4-ket-qua-thuc-nghiem-deepsolo-trocr.ipynb`) |
| **Tìm nhanh trong file** | Trong Jupyter/VS Code/Cursor: **Ctrl+F** → gõ **`NOTE-BUOI4-DEMO-1-ANH`** (hoặc gõ tiêu đề **`Demo cho thầy`**) |
| **Thứ tự kéo xuống** | Sau mục *«Luồng trực quan trong notebook»* và **sau** cell code *«Cấu hình luồng trực quan»* (`MAX_SAMPLES`, `DETECTOR_BACKEND`) → markdown **«Demo cho thầy: một ảnh»** → **cell code ngay dưới** (dòng đầu có `NOTE-BUOI4-DEMO-1-ANH`) — **Run cell đó** sau khi sửa `DEMO_IMAGE`. |
| **Chỗ sửa đường ảnh** | Trong cell code demo: khối comment **`# ========== SỬA TẠI ĐÂY ==========`** — gán `DEMO_IMAGE = Path(r"...")` hoặc để `None` để thử `synth_001.png`. |
| **Sơ đồ TrOCR + ảnh UI** | Ctrl+F **`NOTE-BUOI4-TROCR-PIPELINE-VIZ`** → mục markdown *«TrOCR trên ảnh: Input → …»* + cell code ngay dưới (figure có luồng + overlay chữ). |

---

Repo hiện có 2 luồng để tạo số liệu:

1. **Demo/smoke-test** (`outputs/buoi4/demo`) để kiểm tra metric/report.
2. **Luồng có GT cố định bằng manifest** (`data/test_manifest.csv`) để tính CER/WER/plate accuracy thật trên tập test đã khóa.

**Luồng trực quan trong notebook** (ngay sau phần giới thiệu ngắn): chạy từng bước **localize → crop → tiền xử lý → EasyOCR & TrOCR** trên ảnh trong `data/synthetic_plates` hoặc `data/img`; phần **DeepSolo** đúng nghĩa (train/infer) nằm ngoài repo — có thể **xem trước CSV** đã import và nối vào script đánh giá.

Ngoài ra đã có script xuất lỗi theo từng vị trí ký tự (`province`, `letter`, `serial`) để phục vụ phần phân tích lỗi khi viết báo cáo.

## Kết quả demo hiện tại

Dữ liệu demo gồm 8 mẫu giả lập để kiểm tra pipeline đánh giá.

Cấu hình A - DeepSolo end-to-end:

- Số mẫu: 8
- CER: 0.0469
- WER: 0.3750
- Plate accuracy: 0.6250
- Mean latency: 83.7500 ms

Cấu hình B - DeepSolo + TrOCR:

- Số mẫu: 8
- CER: 0.0156
- WER: 0.1250
- Plate accuracy: 0.8750
- Mean latency: 145.9250 ms

Kết luận tạm thời của demo: cấu hình B chính xác hơn, nhưng chậm hơn.

## Kết quả có GT hiện có trong repo

Hiện repo đã có luồng chạy A/B với **manifest có GT cố định** (`data/test_manifest.csv`) và xuất metric thật:

- `reports/buoi4_ab_metrics.json`
- `reports/buoi4_ab_run_synthetic.md`
- `reports/buoi4_hard_cases.md`

Bộ dữ liệu mẫu đang dùng trong repo là **synthetic** (`data/synthetic_plates/`) để tái lập số liệu trong Git khi chưa commit ảnh thật. Khi có ảnh thực địa + GT, chỉ cần thay nguồn ảnh/nhãn và build lại manifest là giữ nguyên toàn bộ pipeline đánh giá.

## Luồng trực quan trong notebook: localize → crop → OCR

**DeepSolo** trong Buổi 4 là mô hình **text spotting** (định vị vùng chữ, có thể kèm đọc chữ end-to-end). Repo này **không nhúng mã huấn luyện / infer DeepSolo**; khi bạn đã chạy DeepSolo ở repo riêng và export CSV đúng schema, hãy nhập bằng `scripts/run_buoi4_manifest_inference.py` (`--config-a-from-csv` / `--config-b-from-csv`) rồi xem lại metric ở các cell phía dưới.

**Phần dưới đây** chạy **trực tiếp trong notebook** để bạn *nhìn thấy* từng bước thực tế:

1. **Localize** — dùng detector có trong repo: `dummy` (cắt giữa khung, phù hợp ảnh synthetic đã căn sẵn biển) hoặc `yolov8` (cần file `weights/yolov8_license_plate.pt`, phù hợp ảnh thật `data/img`).
2. **Crop + tiền xử lý** — `crop_plate` → `preprocess_plate` (resize, histogram equalize).
3. **Hai nhánh OCR trên cùng một crop** — **EasyOCR** (tương ứng nhánh A trong thí nghiệm manifest) và **TrOCR** (nhánh B).

Chỉnh `SAMPLE_DIR`, `MAX_SAMPLES`, `DETECTOR_BACKEND`, `DEVICE` ở cell code kế tiếp, rồi chạy cell visualization.

**Khi thầy chỉ cho đúng một file ảnh:** xuống mục **«Demo cho thầy: một ảnh»** — chỉ cần gán `DEMO_IMAGE` rồi chạy một cell là có hình.

In [ ]:
# --- Cấu hình luồng trực quan (sửa các biến này rồi chạy lại cell visualization) ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "docs":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MAX_SAMPLES = 4
DETECTOR_BACKEND = "dummy"  # "dummy" | "yolov8"
DEVICE = "cpu"  # "cuda" nếu có GPU + PyTorch CUDA
OCR_GPU = False  # EasyOCR: bật True nếu muốn dùng GPU cho nhánh EasyOCR

# Thư mục ảnh: synthetic (mặc định) hoặc ảnh thật
SAMPLE_DIR = PROJECT_ROOT / "data" / "synthetic_plates"
# SAMPLE_DIR = PROJECT_ROOT / "data" / "img"

YOLO_WEIGHTS = PROJECT_ROOT / "weights" / "yolov8_license_plate.pt"
TROCR_MODEL = "microsoft/trocr-base-printed"

# Map image_id -> GT (nếu có manifest synthetic / test)
MANIFEST_CSV = PROJECT_ROOT / "data" / "test_manifest.csv"

### Demo cho thầy: **một ảnh** bất kỳ

**Mã tìm kiếm (bookmark):** `NOTE-BUOI4-DEMO-1-ANH` — mở notebook, **Ctrl+F**, dán chuỗi này để nhảy thẳng tới cell code demo.

**File đang mở:** `docs/buoi-4-ket-qua-thuc-nghiem-deepsolo-trocr.ipynb` (từ gốc repo `ComputerVisionNew`).

**Vị trí trong notebook (theo thứ tự từ trên xuống):**

1. Markdown *«Luồng trực quan trong notebook: localize → crop → OCR»*
2. Cell code *«Cấu hình luồng trực quan»* (`MAX_SAMPLES`, `SAMPLE_DIR`, …)
3. **← Bạn đang ở đây:** markdown **«Demo cho thầy: một ảnh»**
4. **Cell code ngay dưới dòng này** = cell cần **Run** (đừng nhầm với cell nhiều ảnh phía dưới nữa).

---

Khi thầy yêu cầu mở **một file cụ thể**:

1. Mở **cell code liền kề bên dưới** tiêu đề này.
2. Trong khối **`# ========== SỬA TẠI ĐÂY ==========`**: đặt `DEMO_IMAGE = Path(r"D:\đủ\đường\ảnh.jpg")` hoặc `None` để thử nhanh bằng `data/synthetic_plates/synth_001.png`.
3. Ảnh **toàn cảnh xe** → trước đó chạy cell cấu hình với `DETECTOR_BACKEND = "yolov8"` và có `weights/yolov8_license_plate.pt`. Ảnh **đã crop sát biển** → `dummy` được.
4. (Tùy chọn) `DEMO_GT = "51H12345"` nếu biết nhãn để hiện trên hình.

Cell code **tự đủ** (có thể Run độc lập); nếu đã chạy cell cấu hình, nó tái dùng `DETECTOR_BACKEND` / `DEVICE` / …

> **Note báo cáo:** đây là **detect + crop + EasyOCR/TrOCR** trong repo. **DeepSolo** chạy ngoài repo → nhập CSV + script manifest ở các mục sau.

In [ ]:
# NOTE-BUOI4-DEMO-1-ANH  ← Ctrl+F trong notebook: dán chuỗi này để tìm cell demo 1 ảnh.
# --- Demo 1 ảnh: sửa khối "SỬA TẠI ĐÂY" bên dưới, rồi Run cell này (Shift+Enter) ---
from __future__ import annotations

import csv
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

# Dùng lại biến từ cell cấu hình nếu đã chạy; không thì mặc định dưới đây
try:
    _root = PROJECT_ROOT
except NameError:
    _root = Path.cwd()
    if _root.name == "docs":
        _root = _root.parent

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# ========== SỬA TẠI ĐÂY ==========
# Một file cụ thể thầy đưa (ưu tiên), hoặc để None để demo nhanh bằng synthetic mặc định
DEMO_IMAGE: Path | None = None  # ví dụ: Path(r"D:\ảnh_thầy\plate.jpg")
DEMO_GT = ""  # ví dụ "51H12345" nếu biết nhãn; để "" thì chỉ hiện pred (hoặc lấy từ manifest nếu khớp image_id)

if DEMO_IMAGE is None:
    DEMO_IMAGE = _root / "data" / "synthetic_plates" / "synth_001.png"

try:
    _det = DETECTOR_BACKEND
except NameError:
    _det = "dummy"

try:
    _dev = DEVICE
except NameError:
    _dev = "cpu"

try:
    _ocr_gpu = OCR_GPU
except NameError:
    _ocr_gpu = False

try:
    _trocr = TROCR_MODEL
except NameError:
    _trocr = "microsoft/trocr-base-printed"

_yolo = _root / "weights" / "yolov8_license_plate.pt"
try:
    _yolo = YOLO_WEIGHTS
except NameError:
    pass

try:
    _manifest = MANIFEST_CSV
except NameError:
    _manifest = _root / "data" / "test_manifest.csv"

from src.detector.base import DummyCenterDetector
from src.detector.yolov8_detector import YoloV8PlateDetector
from src.ocr.easyocr_adapter import EasyOcrAdapter
from src.ocr.trocr_adapter import TrOcrAdapter
from src.postprocess.plate_rules import normalize_plate_text, repair_common_ocr_errors
from src.preprocess.ops import crop_plate, preprocess_plate
from src.utils.types import FrameData


def _build_detector():
    if _det == "yolov8":
        if not Path(_yolo).is_file():
            raise FileNotFoundError(f"Chưa có weights: {_yolo}")
        return YoloV8PlateDetector(model_path=Path(_yolo), conf_threshold=0.25)
    return DummyCenterDetector()


def _gt_from_manifest(image_stem: str) -> str:
    if DEMO_GT.strip():
        return DEMO_GT.strip()
    p = Path(_manifest)
    if not p.is_file():
        return ""
    with p.open(encoding="utf-8-sig", newline="") as fp:
        for row in csv.DictReader(fp):
            if (row.get("image_id") or "").strip() == image_stem:
                return (row.get("gt") or "").strip()
    return ""


p = Path(DEMO_IMAGE).expanduser().resolve()
if not p.is_file():
    raise FileNotFoundError(f"Không thấy ảnh: {p}")

bgr = cv2.imread(str(p))
if bgr is None:
    raise RuntimeError(f"cv2.imread không đọc được: {p}")

image_id = p.stem
frame = FrameData(image_id=image_id, frame=bgr, source=str(p))
gt = _gt_from_manifest(image_id)

detector = _build_detector()
easy_ocr = EasyOcrAdapter(gpu=_ocr_gpu)
trocr_ocr = TrOcrAdapter(model_name=_trocr, device=_dev)

dets = detector.predict(frame)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

if not dets:
    axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"{p.name}\n(không detect được biển — thử yolov8 hoặc ảnh crop sát hơn)")
    for ax in axes[1:]:
        ax.axis("off")
else:
    best = max(dets, key=lambda d: d.score)
    vis = bgr.copy()
    x1, y1, x2, y2 = best.bbox_xyxy
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"{p.name}\ndet={best.score:.3f} ({_det})")
    axes[0].axis("off")

    plate_crop = crop_plate(frame, best)
    prepared = preprocess_plate(plate_crop.crop)
    easy_out = easy_ocr.recognize(plate_crop, prepared)
    trocr_out = trocr_ocr.recognize(plate_crop, prepared)
    pred_e = repair_common_ocr_errors(easy_out.text_norm or normalize_plate_text(easy_out.text_raw))
    pred_t = repair_common_ocr_errors(trocr_out.text_norm or normalize_plate_text(trocr_out.text_raw))

    axes[1].imshow(cv2.cvtColor(plate_crop.crop, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Crop")
    axes[1].axis("off")

    axes[2].imshow(prepared, cmap="gray")
    axes[2].set_title(f"EasyOCR: {pred_e!r}\nTrOCR: {pred_t!r}" + (f"\nGT: {gt!r}" if gt else ""))
    axes[2].axis("off")

    print(f"File: {p}")
    print(f"EasyOCR → {pred_e!r} | TrOCR → {pred_t!r}" + (f" | GT: {gt!r}" if gt else ""))

plt.suptitle("Buổi 4 — demo 1 ảnh", fontsize=12)
plt.tight_layout()
plt.show()

## TrOCR trên ảnh: Input → Preprocessing → … → Visualization UI

**Bookmark tìm nhanh:** `NOTE-BUOI4-TROCR-PIPELINE-VIZ` (Ctrl+F trong notebook).

Cell code **ngay dưới** vẽ figure: (1) sơ đồ luồng, (2) ảnh từng bước, (3) **ảnh kết quả có bbox + chữ** nhận dạng.

### Sơ đồ luồng (Mermaid — xem được trên GitHub / một số viewer)

```mermaid
flowchart TD
  A[Input Image] --> B[Preprocessing]
  B --> C[TrOCR Processor]
  C --> D[Vision Transformer Encoder]
  D --> E[Text Decoder]
  E --> F[Recognized Text]
  F --> G[Visualization UI]
```

### Cùng nội dung (ASCII)

```text
Input Image
      ↓
Preprocessing  (crop biển + grayscale + resize + histogram equalize — trong repo)
      ↓
TrOCR Processor  (PIL RGB → pixel_values, Hugging Face)
      ↓
Vision Transformer Encoder  (trong VisionEncoderDecoderModel)
      ↓
Text Decoder  (sinh chuỗi autoregressive, model.generate)
      ↓
Recognized Text  (chuỗi thô → normalize / repair)
      ↓
Visualization UI  (vẽ bbox + overlay chữ lên ảnh gốc)
```

**Ghi chú:** Encoder/Decoder không tách tensor trung gian trong cell (black box); minh họa bằng khung chú thích. Muốn attention map cần hook sâu vào `transformers` (ngoài phạm vi cell demo).

In [ ]:
# NOTE-BUOI4-TROCR-PIPELINE-VIZ — Ctrl+F để tìm cell này. Run sau khi sửa PIPE_IMAGE / detector.
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# ----- Sửa tại đây -----
PIPE_IMAGE = Path(r"d:\ComputerVisionNew\data\img\plate_0029.jpg")
PIPE_DETECTOR = "dummy"  # "dummy" | "yolov8" — nếu đã chạy cell cấu hình, có thể ghi đè bằng DETECTOR_BACKEND
PIPE_DEVICE = "cpu"

# ----- Root project -----
_root = Path.cwd()
if _root.name == "docs":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

try:
    PIPE_DETECTOR = DETECTOR_BACKEND  # type: ignore[name-defined]
except NameError:
    pass
try:
    PIPE_DEVICE = DEVICE  # type: ignore[name-defined]
except NameError:
    pass
try:
    _trocr_name = TROCR_MODEL  # type: ignore[name-defined]
except NameError:
    _trocr_name = "microsoft/trocr-base-printed"

try:
    _yolo = YOLO_WEIGHTS  # type: ignore[name-defined]
except NameError:
    _yolo = _root / "weights" / "yolov8_license_plate.pt"

if PIPE_IMAGE is None:
    PIPE_IMAGE = _root / "data" / "synthetic_plates" / "synth_001.png"

from src.detector.base import DummyCenterDetector
from src.detector.yolov8_detector import YoloV8PlateDetector
from src.ocr.trocr_adapter import TrOcrAdapter
from src.postprocess.plate_rules import normalize_plate_text, repair_common_ocr_errors
from src.preprocess.ops import crop_plate, preprocess_plate
from src.utils.types import FrameData


def _det():
    if PIPE_DETECTOR == "yolov8":
        if not Path(_yolo).is_file():
            raise FileNotFoundError(f"Thiếu weights: {_yolo}")
        return YoloV8PlateDetector(model_path=Path(_yolo), conf_threshold=0.25)
    return DummyCenterDetector()


def _overlay_ui(bgr: np.ndarray, bbox: tuple[int, int, int, int], text: str) -> np.ndarray:
    out = bgr.copy()
    x1, y1, x2, y2 = bbox
    cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thick = 0.85, 2
    (tw, th), bl = cv2.getTextSize(text, font, scale, thick)
    pad = 6
    y0 = min(y2 + th + pad + 8, out.shape[0] - 4)
    x0 = max(0, min(x1, out.shape[1] - tw - 2 * pad))
    cv2.rectangle(out, (x0, y0 - th - 2 * pad), (x0 + tw + 2 * pad, y0 + bl), (20, 20, 20), -1)
    cv2.putText(out, text, (x0 + pad, y0 - pad), font, scale, (0, 255, 100), thick, cv2.LINE_AA)
    return out


p = Path(PIPE_IMAGE).expanduser().resolve()
if not p.is_file():
    raise FileNotFoundError(p)

bgr = cv2.imread(str(p))
if bgr is None:
    raise RuntimeError("cv2.imread thất bại")

image_id = p.stem
frame = FrameData(image_id=image_id, frame=bgr, source=str(p))
detector = _det()
dets = detector.predict(frame)
if not dets:
    raise RuntimeError("Không có detection — thử yolov8 hoặc ảnh crop sát biển.")

best = max(dets, key=lambda d: d.score)
plate_crop = crop_plate(frame, best)
prepared = preprocess_plate(plate_crop.crop)

trocr = TrOcrAdapter(model_name=_trocr_name, device=PIPE_DEVICE)
ocr_out = trocr.recognize(plate_crop, prepared)
recognized = repair_common_ocr_errors(ocr_out.text_norm or normalize_plate_text(ocr_out.text_raw))

ui_bgr = _overlay_ui(bgr, best.bbox_xyxy, recognized)

# ----- Figure: flow + stages + UI -----
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 5, height_ratios=[1.05, 1.15], width_ratios=[0.42, 1.0, 1.0, 1.0, 1.0], hspace=0.28, wspace=0.25)

ax_flow = fig.add_subplot(gs[0, 0])
ax_flow.axis("off")
flow_txt = (
    "Input Image\n"
    "      ↓\n"
    "Preprocessing\n"
    "      ↓\n"
    "TrOCR Processor\n"
    "      ↓\n"
    "ViT Encoder\n"
    "      ↓\n"
    "Text Decoder\n"
    "      ↓\n"
    "Recognized Text\n"
    "      ↓\n"
    "Visualization UI"
)
ax_flow.text(0.5, 0.5, flow_txt, ha="center", va="center", fontsize=9, family="monospace", transform=ax_flow.transAxes)

ax_in = fig.add_subplot(gs[0, 1])
ax_in.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
ax_in.set_title("1) Input Image", fontsize=11)
ax_in.axis("off")

ax_crop = fig.add_subplot(gs[0, 2])
ax_crop.imshow(cv2.cvtColor(plate_crop.crop, cv2.COLOR_BGR2RGB))
ax_crop.set_title("2) Crop (localize)", fontsize=11)
ax_crop.axis("off")

ax_prep = fig.add_subplot(gs[0, 3])
ax_prep.imshow(prepared, cmap="gray")
ax_prep.set_title("3) Preprocessing\n→ vào TrOCR Processor", fontsize=11)
ax_prep.axis("off")

ax_blk = fig.add_subplot(gs[0, 4])
ax_blk.axis("off")
ax_blk.text(
    0.5,
    0.5,
    "4) TrOCR Processor\n   (PIL → pixel_values)\n\n"
    "5) Vision Transformer\n   Encoder (trong HF)\n\n"
    "6) Text Decoder\n   (model.generate)\n\n"
    "— black box —\n"
    "không vẽ tensor ở đây",
    ha="center",
    va="center",
    fontsize=9,
    family="monospace",
    transform=ax_blk.transAxes,
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.85),
)

ax_txt = fig.add_subplot(gs[1, 0])
ax_txt.axis("off")
ax_txt.text(
    0.5,
    0.55,
    "7) Recognized Text\n\n" + repr(recognized),
    ha="center",
    va="center",
    fontsize=14,
    weight="bold",
    family="monospace",
    transform=ax_txt.transAxes,
)

ax_vis = fig.add_subplot(gs[1, 1:])
ax_vis.imshow(cv2.cvtColor(ui_bgr, cv2.COLOR_BGR2RGB))
ax_vis.set_title("8) Visualization UI — bbox + overlay trên ảnh gốc", fontsize=12, weight="bold")
ax_vis.axis("off")

plt.suptitle(f"TrOCR pipeline trên ảnh | {p.name} | det={best.score:.3f} ({PIPE_DETECTOR})", fontsize=13)
plt.show()

print("Recognized:", repr(recognized))
print("Raw OCR:", repr(ocr_out.text_raw))

In [ ]:
# Chạy từng bước: detect -> crop -> preprocess -> EasyOCR & TrOCR, hiển thị hình
from itertools import islice
import csv

import cv2
import matplotlib.pyplot as plt
import numpy as np

from src.detector.base import DummyCenterDetector
from src.detector.yolov8_detector import YoloV8PlateDetector
from src.io.readers import iter_images
from src.ocr.easyocr_adapter import EasyOcrAdapter
from src.ocr.trocr_adapter import TrOcrAdapter
from src.postprocess.plate_rules import normalize_plate_text, repair_common_ocr_errors
from src.preprocess.ops import crop_plate, preprocess_plate

# %matplotlib inline  # bật nếu môi trường Jupyter cần

gt_by_id: dict[str, str] = {}
if MANIFEST_CSV.exists():
    with MANIFEST_CSV.open(encoding="utf-8-sig", newline="") as fp:
        for row in csv.DictReader(fp):
            iid = (row.get("image_id") or "").strip()
            if iid:
                gt_by_id[iid] = (row.get("gt") or "").strip()


def build_detector():
    if DETECTOR_BACKEND == "yolov8":
        if not YOLO_WEIGHTS.is_file():
            raise FileNotFoundError(
                f"Chưa có weights YOLO: {YOLO_WEIGHTS}. Đặt file .pt vào đây hoặc đổi DETECTOR_BACKEND='dummy'."
            )
        return YoloV8PlateDetector(model_path=YOLO_WEIGHTS, conf_threshold=0.25)
    if DETECTOR_BACKEND != "dummy":
        raise ValueError("DETECTOR_BACKEND phải là 'dummy' hoặc 'yolov8'")
    return DummyCenterDetector()


detector = build_detector()
easy_ocr = EasyOcrAdapter(gpu=OCR_GPU)
trocr_ocr = TrOcrAdapter(model_name=TROCR_MODEL, device=DEVICE)

if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError(f"Không thấy thư mục ảnh: {SAMPLE_DIR}")

frames = list(islice(iter_images(SAMPLE_DIR), MAX_SAMPLES))
if not frames:
    raise RuntimeError(f"Không có ảnh hợp lệ trong {SAMPLE_DIR} (kiểm tra đuôi .jpg/.png và cv2.imread).")

fig_h = max(2.5, 2.2 * MAX_SAMPLES)
fig, axes = plt.subplots(MAX_SAMPLES, 3, figsize=(11, fig_h))
if MAX_SAMPLES == 1:
    axes = np.array([axes])

shown = 0
for frame in frames:
    dets = detector.predict(frame)
    ax0, ax1, ax2 = axes[shown]
    vis = frame.frame.copy()
    gt = gt_by_id.get(frame.image_id, "")

    if not dets:
        ax0.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax0.set_title(f"{frame.image_id}\n(không có detection)")
        ax1.axis("off")
        ax2.axis("off")
        shown += 1
        continue

    best = max(dets, key=lambda d: d.score)
    x1, y1, x2, y2 = best.bbox_xyxy
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax0.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax0.set_title(f"{frame.image_id}\ndet score={best.score:.3f}")
    ax0.axis("off")

    plate_crop = crop_plate(frame, best)
    prepared = preprocess_plate(plate_crop.crop)
    easy_out = easy_ocr.recognize(plate_crop, prepared)
    trocr_out = trocr_ocr.recognize(plate_crop, prepared)

    pred_easy = repair_common_ocr_errors(easy_out.text_norm or normalize_plate_text(easy_out.text_raw))
    pred_trocr = repair_common_ocr_errors(trocr_out.text_norm or normalize_plate_text(trocr_out.text_raw))

    ax1.imshow(cv2.cvtColor(plate_crop.crop, cv2.COLOR_BGR2RGB))
    ax1.set_title("Crop (BGR→RGB)")
    ax1.axis("off")

    ax2.imshow(prepared, cmap="gray")
    ax2.set_title(
        "Sau preprocess (320×120)\n"
        f"EasyOCR: {pred_easy!r}\nTrOCR: {pred_trocr!r}"
        + (f"\nGT: {gt!r}" if gt else "")
    )
    ax2.axis("off")

    print(
        f"[{frame.image_id}] det={best.score:.3f} | EasyOCR={pred_easy!r} | TrOCR={pred_trocr!r}"
        + (f" | GT={gt!r}" if gt else "")
    )
    shown += 1

for j in range(shown, MAX_SAMPLES):
    for ax in axes[j]:
        ax.axis("off")

plt.suptitle(
    f"Buổi 4 — bước thực tế: {DETECTOR_BACKEND} → crop → EasyOCR vs TrOCR | {SAMPLE_DIR.name}",
    fontsize=11,
)
plt.tight_layout()
plt.show()

### Khi đã có kết quả DeepSolo (export CSV)

Nếu bạn chạy infer ở repo [DeepSolo](https://github.com/ViTAE-Transformer/DeepSolo) và xuất file CSV cùng cột với `outputs/buoi4/deepsolo_e2e_predictions.csv`, cell code bên dưới chỉ **in vài dòng** để đối chiếu với manifest — phần **đánh giá đầy đủ** dùng lệnh `run_buoi4_manifest_inference.py` với `--config-a-from-csv` / `--config-b-from-csv` (cell phía sau).

In [ ]:
# Xem trước CSV nhập từ DeepSolo (hoặc file A/B do script sinh ra)
import csv
from pathlib import Path

_root = Path.cwd()
if _root.name == "docs":
    _root = _root.parent

for label, rel in [
    ("Config A (e2e / hoặc CSV DeepSolo)", "outputs/buoi4/deepsolo_e2e_predictions.csv"),
    ("Config B (+ TrOCR)", "outputs/buoi4/deepsolo_trocr_predictions.csv"),
]:
    p = _root / rel
    print(f"\n=== {label} ===\n{p}")
    if not p.is_file():
        print("  (chưa có file — chạy run_buoi4_manifest_inference hoặc copy CSV DeepSolo vào đây)")
        continue
    with p.open(encoding="utf-8-sig", newline="") as fp:
        rows = list(csv.DictReader(fp))
    for row in rows[:5]:
        print(dict(row))
    if len(rows) > 5:
        print(f"  ... ({len(rows)} dòng tổng cộng)")

In [ ]:
# Cell chạy đầy đủ luồng Buổi 4 có GT từ manifest cố định.
# Mặc định dùng data/test_manifest.csv (có thể build từ folder ảnh + file GT bằng script khác).
#
# Nếu đã có CSV từ DeepSolo (cột giống outputs/buoi4/deepsolo_e2e_predictions.csv), chèn thêm:
#   "--config-a-from-csv", str(project_root / "path" / "to" / "deepsolo_export.csv"),
# và tương tự --config-b-from-csv nếu muốn nhập nhánh B từ file có sẵn.
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "docs":
    project_root = project_root.parent

run_ab_cmd = [
    sys.executable,
    str(project_root / "scripts" / "run_buoi4_manifest_inference.py"),
    "--manifest",
    str(project_root / "data" / "test_manifest.csv"),
    "--detector-backend",
    "dummy",  # đổi sang yolov8 khi chạy ảnh thật toàn cảnh
    "--device",
    "cpu",
    "--run-metrics",
    "--metrics-json",
    str(project_root / "reports" / "buoi4_ab_metrics.json"),
    "--report-md",
    str(project_root / "reports" / "buoi4_ab_run_synthetic.md"),
    "--export-hard-cases",
    "--hard-cases-md",
    str(project_root / "reports" / "buoi4_hard_cases.md"),
]

print(" ".join(run_ab_cmd))
# Bỏ comment dòng dưới nếu muốn chạy lại full pipeline.
# subprocess.run(run_ab_cmd, check=True)

In [ ]:
# Cell xuất phân tích lỗi theo từng ký tự + vùng (province / letter / serial)
# sau khi đã có prediction CSV của A hoặc B.
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "docs":
    project_root = project_root.parent

char_error_cmd = [
    sys.executable,
    str(project_root / "scripts" / "export_char_errors_csv.py"),
    "--pred-csv",
    str(project_root / "outputs" / "buoi4" / "deepsolo_e2e_predictions.csv"),
    "--output-csv",
    str(project_root / "reports" / "char_errors_by_region.csv"),
]

print(" ".join(char_error_cmd))
# Bỏ comment dòng dưới nếu muốn xuất CSV lỗi theo vị trí ký tự.
# subprocess.run(char_error_cmd, check=True)

## Bộ hình trực quan để trình bày

Các hình dưới đây dùng cho slide Buổi 4. Bạn có thể trình chiếu trực tiếp trong notebook.

### 1) Slide tổng quan kết quả

![Tổng quan Buổi 4](assets/buoi4/overview.png)

### 2) So sánh metric chính

![So sánh metric](assets/buoi4/metrics_comparison.png)

### 3) Trade-off accuracy và latency

![So sánh latency](assets/buoi4/latency_comparison.png)

### 4) Sơ đồ pipeline A/B

![Sơ đồ pipeline](assets/buoi4/pipeline_ab.png)

### 5) Đúng/sai theo từng mẫu

![Đúng sai từng mẫu](assets/buoi4/sample_outcomes.png)

### 6) Kết quả video annotate (EasyOCR vs TrOCR)

- `outputs/video/7733112895099_easyocr_annotated.mp4`
- `outputs/video/7733112895099_trocr_annotated.mp4`

### 7) Bảng so sánh video frame-by-frame

- `reports/video_compare_2videos_1000plus.csv`
- `reports/video_compare_summary_2videos_1000plus.md`

(Bộ trên đã gộp 2 video với tổng 1029 frame.)

In [ ]:
# Chạy cell này nếu bạn muốn sinh lại ảnh trực quan theo kết quả mới nhất (A/B có GT).
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd()
if project_root.name == "docs":
    project_root = project_root.parent

cmd = [
    sys.executable,
    str(project_root / "scripts" / "create_buoi4_visual_report.py"),
    "--metrics-json",
    str(project_root / "reports" / "buoi4_ab_metrics.json"),
    "--config-a-csv",
    str(project_root / "outputs" / "buoi4" / "deepsolo_e2e_predictions.csv"),
    "--config-b-csv",
    str(project_root / "outputs" / "buoi4" / "deepsolo_trocr_predictions.csv"),
    "--output-dir",
    str(project_root / "docs" / "assets" / "buoi4"),
]

print(" ".join(cmd))
# Bỏ comment dòng dưới nếu muốn chạy lại trong notebook.
# subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
import csv
import json

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "docs":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Đường dẫn mặc định sau khi chạy scripts/run_buoi4_manifest_inference.py --run-metrics
metrics_path = PROJECT_ROOT / "reports" / "buoi4_ab_metrics.json"
config_a_csv = PROJECT_ROOT / "outputs" / "buoi4" / "deepsolo_e2e_predictions.csv"
config_b_csv = PROJECT_ROOT / "outputs" / "buoi4" / "deepsolo_trocr_predictions.csv"


def load_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as fp:
        return list(csv.DictReader(fp))


def fmt(value: float) -> str:
    return f"{value:.4f}"


print("Project root:", PROJECT_ROOT)
print("Metrics exists:", metrics_path.exists())
print("Config A exists:", config_a_csv.exists())
print("Config B exists:", config_b_csv.exists())

In [ ]:
with metrics_path.open("r", encoding="utf-8") as fp:
    metrics = json.load(fp)

for key, label in [("config_a", "A - DeepSolo end-to-end"), ("config_b", "B - DeepSolo + TrOCR")]:
    item = metrics[key]
    print(label)
    print("  Số mẫu:", item["num_samples"])
    print("  CER:", fmt(item["cer"]))
    print("  WER:", fmt(item["wer"]))
    print("  Plate accuracy:", fmt(item["plate_accuracy"]))
    print("  Mean latency ms:", fmt(item["mean_latency_ms"]))
    print()

print("Khuyến nghị:", metrics["recommendation"])

## Artifact checklist để trình thầy

Khi trình bày, ưu tiên mở theo thứ tự sau để minh họa rõ từ tổng quan -> định lượng -> định tính:

1. `reports/buoi4_ab_metrics.json` (metric tổng)
2. `reports/buoi4_ab_run_synthetic.md` (nội dung báo cáo A/B)
3. `reports/buoi4_hard_cases.md` (hard cases)
4. `reports/video_compare_summary_2videos_1000plus.md` (so sánh EasyOCR vs TrOCR trên **1029 frame**)
5. `reports/video_compare_2videos_1000plus.csv` (chi tiết frame-by-frame)
6. `outputs/video/7733112895099_easyocr_annotated.mp4`
7. `outputs/video/7733112895099_trocr_annotated.mp4`
8. `outputs/video/7733142959513_easyocr.json` + `outputs/video/7733142959513_trocr.json`

Nếu chạy trên ảnh/video thật của nhóm, chỉ cần thay manifest hoặc file input rồi chạy lại các script tương ứng.

In [ ]:
# Cell in nhanh các số liệu chính để đọc trực tiếp khi trình bày.
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "docs":
    PROJECT_ROOT = PROJECT_ROOT.parent

metrics_json = PROJECT_ROOT / "reports" / "buoi4_ab_metrics.json"
video_summary_md = PROJECT_ROOT / "reports" / "video_compare_summary_2videos_1000plus.md"
video_compare_csv = PROJECT_ROOT / "reports" / "video_compare_2videos_1000plus.csv"

if metrics_json.exists():
    m = json.loads(metrics_json.read_text(encoding="utf-8"))
    print("=== BUOI 4 A/B METRICS (IMAGE/MANIFEST) ===")
    for key, label in [("config_a", "A"), ("config_b", "B")]:
        it = m[key]
        print(
            f"{label}: samples={it['num_samples']}, CER={it['cer']:.4f}, "
            f"WER={it['wer']:.4f}, Acc={it['plate_accuracy']:.4f}, Lat={it['mean_latency_ms']:.2f} ms"
        )
    print("Recommendation:", m.get("recommendation", ""))
else:
    print("Chưa có file:", metrics_json)

print("\n=== VIDEO 1000+ FRAMES ===")
print("Summary:", video_summary_md)
print("Detail CSV:", video_compare_csv)
if video_summary_md.exists():
    print(video_summary_md.read_text(encoding="utf-8")[:1200])
else:
    print("Chưa có video summary 1000+ frame, chạy compare + summarize scripts trước.")

In [ ]:
pred_a = load_csv(config_a_csv)
pred_b = load_csv(config_b_csv)

print("Prediction A:")
for row in pred_a:
    print(row)

print("\nPrediction B:")
for row in pred_b:
    print(row)

In [ ]:
pred_b_by_id = {row["image_id"]: row for row in pred_b}
comparison = []
for row_a in pred_a:
    row_b = pred_b_by_id.get(row_a["image_id"], {})
    comparison.append(
        {
            "image_id": row_a["image_id"],
            "gt": row_a["gt"],
            "pred_a": row_a["pred"],
            "pred_b": row_b.get("pred", ""),
            "a_correct": row_a["gt"] == row_a["pred"],
            "b_correct": row_a["gt"] == row_b.get("pred", ""),
        }
    )

for row in comparison:
    print(row)

In [ ]:
recommendation = metrics["recommendation"]
fairness = metrics["fairness"]

print("Kiểm tra cùng test set:", fairness["same_image_ids"])
print("Số ảnh chung:", fairness["common_image_count"])
print("Khuyến nghị:", recommendation)

## Chạy lại khi có ảnh + GT thật

Bước chuẩn hiện tại:

1. Từ thư mục ảnh + file GT, build manifest:

```bash
python scripts/build_test_manifest_from_folder.py --images-dir <thu_muc_anh> --labels-csv <file_gt.csv> --output data/test_manifest.csv
```

2. Chạy A/B + metric:

```bash
python scripts/run_buoi4_manifest_inference.py --manifest data/test_manifest.csv --detector-backend yolov8 --run-metrics
```

3. (Tùy chọn) Xuất lỗi theo vị trí ký tự:

```bash
python scripts/export_char_errors_csv.py --pred-csv outputs/buoi4/deepsolo_e2e_predictions.csv --output-csv reports/char_errors_by_region.csv
```

Schema prediction vẫn giữ:

```csv
image_id,gt,pred,score,latency_ms,bbox_xyxy,error_type
```

In [ ]:
# Command mẫu để chạy full từ manifest hiện tại (đã có GT)
real_config_a_csv = PROJECT_ROOT / "outputs" / "buoi4" / "deepsolo_e2e_predictions.csv"
real_config_b_csv = PROJECT_ROOT / "outputs" / "buoi4" / "deepsolo_trocr_predictions.csv"
manifest_csv = PROJECT_ROOT / "data" / "test_manifest.csv"

print("Manifest:", manifest_csv)
print("Config A CSV:", real_config_a_csv)
print("Config B CSV:", real_config_b_csv)
print("Run trong terminal:")
print(
    "python scripts/run_buoi4_manifest_inference.py "
    "--manifest data/test_manifest.csv "
    "--detector-backend yolov8 "
    "--run-metrics "
    "--metrics-json reports/buoi4_ab_metrics.json "
    "--report-md reports/buoi4_ab_run_synthetic.md"
)

---

# Phụ lục học nhanh: hiểu Buổi 4 từ mất gốc

Buổi 4 không chỉ chạy model, mà là **so sánh hai cách nhận diện biển số**:

- **Cấu hình A - DeepSolo end-to-end**: một hệ thống cố gắng phát hiện vùng chữ/biển số và đọc chữ trong cùng một hướng tiếp cận.
- **Cấu hình B - DeepSolo + TrOCR**: DeepSolo/Detector phụ trách tìm hoặc crop vùng biển số, sau đó TrOCR đọc chữ trong crop đó.

Mục tiêu của Buổi 4 là trả lời câu hỏi:

> Cấu hình nào đọc biển số chính xác hơn, và cái giá phải trả về tốc độ là bao nhiêu?

Pipeline tổng quát:

$$
\text{Ảnh xe} \rightarrow \text{Detect/Crop biển số} \rightarrow \text{OCR đọc chữ} \rightarrow \text{Postprocess} \rightarrow \text{Đánh giá CER/WER/Accuracy/Latency}
$$

Notebook hiện có hai loại kết quả:

1. **Kết quả demo A/B**: có ground truth giả lập để tính CER, WER, plate accuracy.
2. **Kết quả inference thật**: chạy trên ảnh thật trong repo, nhưng chưa có `text_gt`, nên chưa tính accuracy thật được.

## Thuật toán và mô hình được dùng trong Buổi 4

### 1) DeepSolo có ý nghĩa gì?

DeepSolo là hướng tiếp cận nhận dạng chữ trong ảnh tự nhiên. Trong bối cảnh dự án biển số, có thể hiểu đơn giản:

$$
\text{Ảnh} \rightarrow \text{tìm vùng chữ/biển số} \rightarrow \text{dự đoán chuỗi ký tự}
$$

Ý nghĩa trong dự án:

- Giúp xử lý biển số như một vùng text trong ảnh.
- Có thể dùng theo hướng end-to-end: từ ảnh ra text.
- Nếu vùng biển số nhỏ, nghiêng, mờ, hoặc bị phản sáng, DeepSolo/Detector có thể crop chưa tốt, làm OCR sai.

### 2) TrOCR có ý nghĩa gì?

TrOCR là OCR dựa trên Transformer. Hiểu đơn giản:

$$
\text{Ảnh crop biển số} \rightarrow \text{Encoder nhìn ảnh} \rightarrow \text{Decoder sinh từng ký tự}
$$

Cơ chế gần giống dịch máy:

$$
P(y|x) = \prod_{t=1}^{T} P(y_t \mid y_{<t}, x)
$$

Trong đó:

- $x$ là ảnh crop biển số.
- $y_t$ là ký tự thứ $t$ được sinh ra.
- $y_{<t}$ là các ký tự đã sinh trước đó.

Ý nghĩa trong dự án:

- TrOCR thường đọc chữ tốt hơn nếu crop biển số đã sạch.
- Đổi lại, TrOCR có thể chậm hơn vì phải sinh chuỗi ký tự tuần tự.

### 3) Vì sao so sánh A/B?

A/B testing nghĩa là giữ cùng tập ảnh, rồi chạy hai cấu hình khác nhau:

$$
\text{Cùng test set} \rightarrow \begin{cases}
A: \text{DeepSolo end-to-end} \\
B: \text{DeepSolo + TrOCR}
\end{cases} \rightarrow \text{So metric}
$$

Điều kiện so sánh công bằng:

- Hai cấu hình phải chạy trên cùng danh sách `image_id`.
- Cùng ground truth `gt`.
- Cùng quy tắc normalize text trước khi tính lỗi.
- Cùng cách đo latency.

Trong notebook hiện tại, `same_image_ids = True` và số ảnh chung là 8, nên demo A/B đang công bằng về mặt danh sách mẫu.

## Công thức toán học và ý nghĩa trong dự án

### 1) Levenshtein distance - nền tảng của CER/WER

CER và WER đều dựa trên khoảng cách chỉnh sửa Levenshtein:

$$
d(a,b) = \text{số thao tác ít nhất để biến chuỗi } a \text{ thành chuỗi } b
$$

Các thao tác gồm:

- Thêm ký tự/token.
- Xóa ký tự/token.
- Thay ký tự/token.

Ví dụ:

$$
GT = 43B67890,\quad Pred = 43B6789O
$$

Sai khác ở ký tự cuối `0` và `O`, nên khoảng cách ký tự là 1.

### 2) CER - Character Error Rate

$$
CER = \frac{\sum_i EditDistance(chars(GT_i), chars(Pred_i))}{\sum_i |chars(GT_i)|}
$$

Ý nghĩa trong dự án:

- Đo lỗi ở cấp ký tự.
- Rất quan trọng với biển số vì chỉ sai 1 ký tự cũng có thể thành biển số khác.
- CER càng thấp càng tốt.

Ví dụ nếu biển số có 8 ký tự và sai 1 ký tự:

$$
CER = \frac{1}{8} = 0.125
$$

### 3) WER - Word Error Rate

$$
WER = \frac{\sum_i EditDistance(tokens(GT_i), tokens(Pred_i))}{\sum_i |tokens(GT_i)|}
$$

Trong dự án biển số, mỗi biển số thường được xem như một token nếu không tách dòng/tách khoảng trắng.

Ý nghĩa:

- Nếu cả chuỗi biển số sai dù chỉ một ký tự, WER có thể tính là sai cả token.
- WER thường nghiêm khắc hơn trong bài toán biển số ngắn.

### 4) Plate accuracy

$$
PlateAccuracy = \frac{\#\{i: normalize(GT_i) = normalize(Pred_i)\}}{N}
$$

Ý nghĩa trong dự án:

- Đây là chỉ số dễ hiểu nhất: bao nhiêu biển số được đọc đúng toàn bộ.
- Nếu sai 1 ký tự, mẫu đó bị tính là sai.
- Plate accuracy càng cao càng tốt.

### 5) Mean latency

$$
MeanLatency = \frac{1}{N}\sum_{i=1}^{N} latency_i
$$

Ý nghĩa:

- Đo thời gian xử lý trung bình mỗi ảnh.
- Latency càng thấp càng tốt nếu muốn chạy realtime/webcam.
- Nhưng thường có trade-off: mô hình chính xác hơn có thể chậm hơn.

### 6) Cách đọc kết quả demo hiện tại

Cấu hình A:

$$
CER_A = 0.0469,\quad WER_A = 0.3750,\quad Acc_A = 0.6250,\quad Latency_A = 83.75ms
$$

Cấu hình B:

$$
CER_B = 0.0156,\quad WER_B = 0.1250,\quad Acc_B = 0.8750,\quad Latency_B = 145.925ms
$$

Diễn giải:

- B chính xác hơn: CER thấp hơn, WER thấp hơn, plate accuracy cao hơn.
- B chậm hơn: latency trung bình cao hơn A.
- Nếu ưu tiên độ chính xác báo cáo, chọn B.
- Nếu ưu tiên tốc độ realtime, cần cân nhắc A hoặc tối ưu B.

In [ ]:
# Sơ đồ luồng A/B Buổi 4.
# Hình chỉ được tạo khi chạy cell, không nhúng sẵn output vào notebook.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")


def draw_box(x, y, w, h, title, body, color="#EEF5FF"):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03,rounding_size=0.06",
        linewidth=1.5,
        edgecolor="#4C78A8",
        facecolor=color,
    )
    ax.add_patch(box)
    ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(x + w / 2, y + h * 0.34, body, ha="center", va="center", fontsize=9)


def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.6))

# Shared input

draw_box(0.3, 2.35, 1.6, 0.9, "Ảnh đầu vào", "Ảnh xe / biển số", "#F7F7F7")
arrow(1.9, 2.8, 2.6, 4.25)
arrow(1.9, 2.8, 2.6, 1.35)

# Config A

draw_box(2.6, 3.8, 1.8, 0.9, "A: DeepSolo", "End-to-end\nảnh -> text", "#EEF5FF")
arrow(4.4, 4.25, 5.2, 4.25)
draw_box(5.2, 3.8, 1.7, 0.9, "Postprocess", "Normalize text", "#EEF5FF")
arrow(6.9, 4.25, 7.7, 4.25)
draw_box(7.7, 3.8, 1.8, 0.9, "Đánh giá A", "CER/WER/Acc\nLatency", "#EEF5FF")

# Config B

draw_box(2.6, 0.9, 1.8, 0.9, "B: Detector", "Tìm/crop\nbiển số", "#F2FFF0")
arrow(4.4, 1.35, 5.2, 1.35)
draw_box(5.2, 0.9, 1.7, 0.9, "TrOCR", "Crop -> chuỗi ký tự", "#F2FFF0")
arrow(6.9, 1.35, 7.7, 1.35)
draw_box(7.7, 0.9, 1.8, 0.9, "Đánh giá B", "CER/WER/Acc\nLatency", "#F2FFF0")

# Comparison
arrow(8.6, 3.8, 8.6, 1.8)
draw_box(3.8, 2.35, 2.6, 0.75, "So sánh công bằng", "Cùng image_id + cùng ground truth", "#FFF8E8")

ax.set_title("Luồng so sánh A/B: DeepSolo end-to-end vs DeepSolo + TrOCR", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Biểu đồ trade-off accuracy và latency.
# Ưu tiên dùng biến metrics nếu đã chạy cell đọc JSON; nếu chưa thì dùng số liệu demo.
import matplotlib.pyplot as plt

fallback_metrics = {
    "config_a": {
        "cer": 0.046875,
        "wer": 0.375,
        "plate_accuracy": 0.625,
        "mean_latency_ms": 83.75,
        "error_counts": {"ok": 5, "ocr_or_spotting": 3},
    },
    "config_b": {
        "cer": 0.015625,
        "wer": 0.125,
        "plate_accuracy": 0.875,
        "mean_latency_ms": 145.925,
        "error_counts": {"ok": 7, "ocr_or_spotting": 1},
    },
}

metrics_for_plot = metrics if "metrics" in globals() else fallback_metrics
config_names = ["A\nDeepSolo", "B\nDeepSolo+TrOCR"]
cer_values = [metrics_for_plot["config_a"]["cer"], metrics_for_plot["config_b"]["cer"]]
wer_values = [metrics_for_plot["config_a"]["wer"], metrics_for_plot["config_b"]["wer"]]
acc_values = [metrics_for_plot["config_a"]["plate_accuracy"], metrics_for_plot["config_b"]["plate_accuracy"]]
latency_values = [metrics_for_plot["config_a"]["mean_latency_ms"], metrics_for_plot["config_b"]["mean_latency_ms"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(config_names, acc_values, color=["#4E79A7", "#59A14F"])
axes[0].set_ylim(0, 1)
axes[0].set_title("Plate accuracy\n(càng cao càng tốt)")
for i, v in enumerate(acc_values):
    axes[0].text(i, v, f"{v:.3f}", ha="center", va="bottom")

x = range(len(config_names))
axes[1].bar([i - 0.18 for i in x], cer_values, width=0.36, label="CER", color="#E15759")
axes[1].bar([i + 0.18 for i in x], wer_values, width=0.36, label="WER", color="#F28E2B")
axes[1].set_xticks(list(x), config_names)
axes[1].set_ylim(0, max(wer_values + cer_values) * 1.25)
axes[1].set_title("Lỗi OCR\n(càng thấp càng tốt)")
axes[1].legend()

axes[2].bar(config_names, latency_values, color=["#76B7B2", "#B07AA1"])
axes[2].set_title("Mean latency ms\n(càng thấp càng nhanh)")
for i, v in enumerate(latency_values):
    axes[2].text(i, v, f"{v:.1f} ms", ha="center", va="bottom")

plt.tight_layout()
plt.show()

print("Nhận xét nhanh:")
print("- B chính xác hơn vì plate accuracy cao hơn và CER/WER thấp hơn.")
print("- B chậm hơn vì mean latency cao hơn.")
print("- Chọn B nếu ưu tiên độ chính xác; cân nhắc tối ưu B nếu cần realtime.")

## Kết quả Buổi 4 sẽ được đánh giá như thế nào?

Đánh giá Buổi 4 nên nhìn theo 5 lớp: **công bằng dữ liệu**, **độ chính xác OCR**, **tốc độ**, **phân tích lỗi**, và **định tính video thực tế**.

### 1) Kiểm tra công bằng A/B

Trước khi so metric, phải đảm bảo hai cấu hình chạy trên cùng tập ảnh:

$$
IDs_A = IDs_B
$$

Trong pipeline hiện tại, trạng thái này nằm trong `fairness` của file `reports/buoi4_ab_metrics.json`.

### 2) Đánh giá độ chính xác OCR

Ưu tiên đọc theo thứ tự:

1. `plate_accuracy`
2. `CER`
3. `WER`
4. `error_type`

Với biển số, chỉ sai 1 ký tự đã có thể thành biển khác nên `plate_accuracy` là chỉ số quan trọng nhất khi báo cáo.

### 3) Đánh giá tốc độ

So sánh `mean_latency_ms` của A/B trên cùng thiết lập phần cứng.

Ngoài ảnh tĩnh, đã có luồng video:

- JSON theo frame: `outputs/video/*_easyocr*.json`, `outputs/video/*_trocr*.json`
- Bảng so sánh frame-by-frame (2 video, 1029 frame): `reports/video_compare_2videos_1000plus.csv`
- Bảng tóm tắt để dán báo cáo: `reports/video_compare_summary_2videos_1000plus.md`

### 4) Đánh giá phân tích lỗi

Nhóm lỗi chính hiện dùng:

- `detect_miss`
- `bad_crop`
- `ocr_error`
- `postprocess_helped`
- `ambiguous_gt`

Ngoài ra có phân tích lỗi theo **vị trí ký tự** (`province`, `letter`, `serial`) bằng script `export_char_errors_csv.py`.

### 5) Kết luận và giới hạn

Kết luận nên dựa trên số liệu từ manifest có GT cố định. Nếu đang dùng tập synthetic hoặc frame sampling ngắn, cần ghi rõ giới hạn và kế hoạch mở rộng sang ảnh/video thực địa để tăng tính thuyết phục.

---

## File code liên quan đến notebook Buổi 4

Notebook này dùng để xem kết quả thực nghiệm A/B giữa DeepSolo end-to-end và DeepSolo + TrOCR. Logic chính nằm ở các file sau:

### 1) Tạo dữ liệu test + chạy A/B metrics

- [`../scripts/generate_synthetic_plate_dataset.py`](../scripts/generate_synthetic_plate_dataset.py): tạo ảnh + `data/test_manifest.csv` mẫu.
- [`../scripts/build_test_manifest_from_folder.py`](../scripts/build_test_manifest_from_folder.py): ghép `data/test_manifest.csv` từ folder ảnh + file GT (CSV/JSON/txt).
- [`../scripts/run_buoi4_manifest_inference.py`](../scripts/run_buoi4_manifest_inference.py): chạy A/B từ manifest, xuất prediction CSV, metrics và hard cases.
- [`../scripts/run_buoi4_experiments.py`](../scripts/run_buoi4_experiments.py): tính CER/WER/plate accuracy/latency từ 2 CSV prediction.

### 2) Metric và phân tích lỗi OCR/biển số

- [`../src/eval/metrics_plate.py`](../src/eval/metrics_plate.py): Levenshtein, `cer`, `wer`, `plate_accuracy`.
- [`../src/eval/error_labels.py`](../src/eval/error_labels.py): gán `detect_miss`, `bad_crop`, `ocr_error`, `postprocess_helped`, `ambiguous_gt`.
- [`../src/eval/char_error_regions.py`](../src/eval/char_error_regions.py): lỗi theo vị trí ký tự (`province`, `letter`, `serial`).
- [`../scripts/export_char_errors_csv.py`](../scripts/export_char_errors_csv.py): xuất CSV lỗi ký tự theo vùng từ file prediction.

### 3) Inference và OCR backend

- [`../src/pipeline/infer_plate_pipeline.py`](../src/pipeline/infer_plate_pipeline.py): pipeline chính `detector -> crop -> preprocess -> OCR -> postprocess`.
- [`../src/pipeline/detailed_plate_infer.py`](../src/pipeline/detailed_plate_infer.py): inference chi tiết để phục vụ phân loại lỗi.
- [`../src/detector/yolov8_detector.py`](../src/detector/yolov8_detector.py): detector YOLOv8.
- [`../src/ocr/easyocr_adapter.py`](../src/ocr/easyocr_adapter.py): OCR EasyOCR.
- [`../src/ocr/trocr_adapter.py`](../src/ocr/trocr_adapter.py): OCR TrOCR.

### 4) Chuẩn bị DeepSolo và TrOCR

- [`../scripts/prepare_buoi4_deepsolo_data.py`](../scripts/prepare_buoi4_deepsolo_data.py): convert manifest sang format annotation cho nhánh DeepSolo.
- [`../configs/deepsolo/README.md`](../configs/deepsolo/README.md): hướng dẫn cấu hình DeepSolo (repo ngoài).
- [`../configs/trocr/README.md`](../configs/trocr/README.md): hướng dẫn TrOCR và fine-tune.

### 5) Nên đọc theo thứ tự

1. `scripts/run_buoi4_manifest_inference.py`
2. `scripts/run_buoi4_experiments.py`
3. `src/eval/metrics_plate.py` + `src/eval/char_error_regions.py`
4. `src/pipeline/infer_plate_pipeline.py` + `src/pipeline/detailed_plate_infer.py`
5. `src/ocr/trocr_adapter.py` và `src/ocr/easyocr_adapter.py`